In [ ]:
import os
from ultralytics import YOLO
import matplotlib.pyplot as plt
import cv2

# 1. Defina o dataset que será validado neste momento (Ex: 'placas', 'ingredientes')
DATASET_ATIVO = "placas" 
ARQUIVO_CONFIG = f"../data_{DATASET_ATIVO}.yaml"

# 2. Carrega o modelo (Aponte para o modelo padrão ou para o seu customizado após o treino)
# MODELO_PATH = "../models/yolov8s.pt" # Baseline padrão
MODELO_PATH = f"../models/treinamentos/yolo_{DATASET_ATIVO}/weights/best.pt"

print(f"Carregando o modelo: {MODELO_PATH}")
model = YOLO(MODELO_PATH)

## TESTE 1 – Inferência em uma única Imagem Individual

In [ ]:
# Defina o caminho de uma imagem de teste avulsa
IMAGEM_TESTE = "../tests/foto_rua_carros.jpg"

# Executa a inferência
resultados = model(IMAGEM_TESTE)

# Mostra as saídas brutas estruturadas no terminal/célula
print("\n--- DETECÇÕES ESTRUTURADAS (SAÍDA JSON DA API) ---")
for result in resultados:
    boxes = result.boxes
    for box in boxes:
        conf = float(box.conf)
        cls_id = int(box.cls)
        nome_classe = model.names[cls_id]
        
        if conf > 0.4:  # Filtro de confiança mínima
            x_min, y_min, x_max, y_max = box.xyxy.tolist()[0]
            print(f"Classe: {nome_classe} | Confiança: {conf:.2f} | BBox: [{int(x_min)}, {int(y_min)}, {int(x_max)}, {int(y_max)}]")

# Exibe o resultado visual desenhado diretamente na tela do Notebook
print("\n--- VISUALIZAÇÃO GRÁFICA ---")
imagem_desenhada = resultados[0].plot()
imagem_rgb = cv2.cvtColor(imagem_desenhada, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 6))
plt.imshow(imagem_rgb)
plt.axis('off')
plt.show()

In [ ]:
# Executa a validação oficial do YOLO no dataset ativo
metrics = model.val(data=ARQUIVO_CONFIG, split="val", plots=False)

# Altere o nome abaixo para a classe exata que você deseja inspecionar
CLASSE_ALVO = "roda" 

# Descobre o ID correspondente ao nome da classe alvo
id_classe_alvo = None
for idx, name in model.names.items():
    if name == CLASSE_ALVO:
        id_classe_alvo = idx
        break

if id_classe_alvo is not None:
    # Extrai as métricas de Precision, Recall e mAP50 para a classe alvo
    precision_classe = metrics.precision[id_classe_alvo]
    recall_classe = metrics.recall[id_classe_alvo]
    map50_classe = metrics.map50[id_classe_alvo]
    
    print(f"=== MÉTRICAS ISOLADAS PARA A CLASSE: '{CLASSE_ALVO.upper()}' ===")
    print(f"Precisão (Precision): {precision_classe:.4f}")
    print(f"Revocação (Recall):    {recall_classe:.4f}")
    print(f"mAP50 da Classe:       {map50_classe:.4f}")
else:
    print(f"Classe '{CLASSE_ALVO}' não encontrada no mapeamento do arquivo yaml.")

In [ ]:
import pandas as pd

# Executa novamente habilitando a geração dos gráficos visuais
print("Processando todas as classes para gerar o relatório final...")
metrics_globais = model.val(data=ARQUIVO_CONFIG, split="val", plots=True)

# 1. Constrói o Classification Report estruturado em tabela
classes_mapeadas = list(model.names.values())
report_data = {
    "Classe": classes_mapeadas,
    "Precisão (Precision)": [float(p) for p in metrics_globais.precision],
    "Revocação (Recall)": [float(r) for r in metrics_globais.recall],
    "mAP50": [float(m) for m in metrics_globais.map50],
    "mAP50-95": [float(m) for m in metrics_globais.map95]
}

df_report = pd.DataFrame(report_data)

# 2. Adiciona a linha de média global (Macro Average)
linha_media = pd.DataFrame([{
    "Classe": "MÉDIA GLOBAL (all)",
    "Precisão (Precision)": float(metrics_globais.box.mp),
    "Revocação (Recall)": float(metrics_globais.box.mr),
    "mAP50": float(metrics_globais.box.map50),
    "mAP50-95": float(metrics_globais.box.map)
}])
df_report = pd.concat([df_report, linha_media], ignore_index=True)

# Exibe o relatório tabular estilizado
pd.set_option('display.float_format', lambda x: '%.4f' % x)
print("\n======================= CLASSIFICATION REPORT (YOLO) =======================")
display(df_report)

# 3. Exibe o gráfico da Matriz de Confusão para fechar a análise do vídeo
caminho_matriz = os.path.join(metrics_globais.save_dir, "confusion_matrix.png")
if os.path.exists(caminho_matriz):
    print("\n--- MATRIZ DE CONFUSÃO DO DATASET ---")
    matriz_img = cv2.imread(caminho_matriz)
    matriz_rgb = cv2.cvtColor(matriz_img, cv2.COLOR_BGR2RGB)
    
    plt.figure(figsize=(11, 9))
    plt.imshow(matriz_rgb)
    plt.axis('off')
    plt.show()